### Imports

In [1]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from scipy.stats import spearmanr
from scipy.stats import entropy as scipy_entropy

import torch
import torch.nn as nn
import torch.nn.init as init

from captum.attr import LRP, LayerLRP

### build 30_features csv


In [ ]:
CSV_PATH = "Dataset/CSV/paired_data_datscan.csv"

MEDICAL_TOP30_FEATURES = [
    'NP3RISNG', 'NP3FTAPR', 'NP3RIGLU', 'NP3TOT',   'DRMREMEM',
    'NP3RIGN',  'SCAU13',   'SCAU4',    'NP2PTOT',  'PARKISM',
    'ESS5',     'NP2SALV',  'HRPOSTMED','SCAU8',    'SCAU23A',
    'PDTRTMNT', 'NP3RIGRU', 'SCAU26D',  'SCAU26A',  'DRMVERBL',
    'SCAU22',   'NP2EAT',   'SCAU12',   'DRMVIVID', 'DRMNOCTB',
    'SLPDSTRB', 'SCAU26C',  'ESS4',     'NP1COG',   'SCAU14'
]

# Columns to keep
LABEL_COLUMN = "NHY"
ID_COLUMNS = ["PATNO", "EVENT_ID", "Image Data ID"]
selected_columns = ID_COLUMNS + MEDICAL_TOP30_FEATURES + [LABEL_COLUMN]

df = pd.read_csv(CSV_PATH)

df_filtered = df[selected_columns]

output_path = "Dataset/CSV/paired_data_datscan_30_features.csv"
df_filtered.to_csv(output_path, index=False)

print("Saved:", output_path)

### Config + Model + Data

In [ ]:
# CONFIG
MODEL_PATH = "models/MedicalRecordProcessor_WEIGHTS.pth"
CSV_PATH = "Dataset/CSV/paired_data_datscan_30_features.csv"

LABEL_COLUMN = "NHY"
ID_COLUMNS = ["PATNO", "EVENT_ID", "Image Data ID"]

TOP_N_GLOBAL = 10    # number of top global features to plot
TOP_N_PER_CLASS = 7  # number of top features per Parkinson stage
TOP_N_FEATURES = 5   # number of top features to include in stage_progression
TOP_N_STAGE   = 8    # features per stage for push-pull graph


OUTPUT_FOLDER = "lrp_analysis"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)



# MODEL DEFINITION; 
class MedicalRecordProcessor(nn.Module):
    """ 30 -> 128 -> 256 -> 5 """
    def __init__(self, num_features=30, num_classes=5):
        super().__init__()
        self.num_features = num_features
        self.num_classes = num_classes

        self.mlp = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),

            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
        )

        self.classifier = nn.Linear(256, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        """He initialization for all linear layers"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x, classify=True):
        if x.shape[1] != self.num_features:
            raise ValueError(f"Expected {self.num_features} features, got {x.shape[1]}")

        features = self.mlp(x)

        if classify:
            return self.classifier(features)
        return features



# LOAD MODEL
device = torch.device("cpu")
model = MedicalRecordProcessor()
checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()



# LOAD DATA
df = pd.read_csv(CSV_PATH)

labels = df[LABEL_COLUMN].values.astype(int)

id_df = df[ID_COLUMNS]  # keep IDs to save later

features_df = df.drop(columns=[LABEL_COLUMN] + ID_COLUMNS, errors='ignore')
feature_names = features_df.columns.tolist()
X = features_df.values.astype(np.float32)
X_tensor = torch.tensor(X, dtype=torch.float32)



# helper for plotting
def plot_top_features(df_sorted, n,save_path, title=""):
    top_df = df_sorted.head(n)
    plt.figure(figsize=(8, 5))
    plt.barh(top_df["feature"], top_df["mean_abs_relevance"])
    plt.gca().invert_yaxis()
    plt.title(title)
    plt.xlabel("Mean Absolute Relevance")
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

### LRP implementation

In [13]:
# LRP WRAPPER (removes BatchNorm for compatibility)
class LRPMLPWrapper(nn.Module):
    def __init__(self, mlp, classifier):
        super().__init__()
        self.layers = nn.Sequential(*(l for l in mlp if not isinstance(l, nn.BatchNorm1d)))
        self.classifier = classifier

    def forward(self, x):
        return self.classifier(self.layers(x))

lrp_model = LRPMLPWrapper(model.mlp, model.classifier)
lrp_model.eval()

layers_to_explain = [
    ("layer1_linear", lrp_model.layers[0]),
    ("layer2_linear", lrp_model.layers[3]),
    ("classifier", lrp_model.classifier),
]
layer_lrps = {name: LayerLRP(lrp_model, layer) for name, layer in layers_to_explain}
layer_outputs = {name: [] for name, _ in layers_to_explain}
lrp = LRP(lrp_model)


# LRP COMPUTATION
all_relevance = []
pred_classes  = []

for i in range(X_tensor.shape[0]):
    x = X_tensor[i].unsqueeze(0)
    pred_class = lrp_model(x).argmax(dim=1).item()
    pred_classes.append(pred_class)

    relevance = lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
    all_relevance.append(relevance)

    for layer_name, layer_lrp in layer_lrps.items():
        layer_rel = layer_lrp.attribute(x, target=pred_class).detach().cpu().numpy().flatten()
        layer_outputs[layer_name].append(layer_rel)

all_relevance = np.array(all_relevance)
pred_classes = np.array(pred_classes)
correct_mask = pred_classes == labels
unique_stages = sorted(np.unique(labels))
n_stages = len(unique_stages)

global_mean_signed = all_relevance.mean(axis=0)
global_mean_abs = np.abs(all_relevance).mean(axis=0)

global_df = pd.DataFrame({
    "feature": feature_names,
    "mean_signed_relevance": global_mean_signed,
    "mean_abs_relevance": global_mean_abs,
}).sort_values("mean_abs_relevance", ascending=False)

stage_cmap = plt.colormaps['tab10'].resampled(n_stages)
stage_colors = {s: stage_cmap(i) for i, s in enumerate(unique_stages)}


# LAYER RELEVANCE
layer_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance")
os.makedirs(layer_folder, exist_ok=True)

for layer_name, values in layer_outputs.items():
    layer_array = np.array(values)
    cur_folder = os.path.join(layer_folder, layer_name)
    os.makedirs(cur_folder, exist_ok=True)

    layer_df = pd.DataFrame(layer_array)
    layer_df = pd.concat([id_df.reset_index(drop=True), layer_df], axis=1)
    layer_df.to_csv(os.path.join(cur_folder, "relevance_per_patient.csv"), index=False)

    median_relevance = np.median(layer_array, axis=0)
    pd.DataFrame({
        "neuron_index": range(len(median_relevance)),
        "median_relevance": median_relevance,
    }).to_csv(os.path.join(cur_folder, "median_relevance.csv"), index=False)


# PER PATIENT RELEVANCE
per_patient_df = pd.DataFrame(all_relevance, columns=feature_names)
per_patient_df = pd.concat([id_df.reset_index(drop=True), per_patient_df], axis=1)
per_patient_df[LABEL_COLUMN] = labels
per_patient_df["predicted_class"] = pred_classes
per_patient_df["correct"] = correct_mask
per_patient_df.to_csv(f"{OUTPUT_FOLDER}/relevance_per_patient.csv", index=False)


# GLOBAL IMPORTANCE
global_folder = os.path.join(OUTPUT_FOLDER, "global_importance")
os.makedirs(global_folder, exist_ok=True)

global_df.to_csv(f"{global_folder}/global_feature_importance.csv", index=False)
 
plot_top_features(global_df, TOP_N_GLOBAL, f"{global_folder}/top_global_features.png" , f"Global Feature Importance - Top {TOP_N_GLOBAL} Features")
 
sorted_abs = global_df.sort_values("mean_abs_relevance", ascending=True)
vmax_signed = sorted_abs["mean_signed_relevance"].abs().max()
fig, ax = plt.subplots(figsize=(8, max(6, len(feature_names) * 0.22)))
sc = ax.scatter(
    sorted_abs["mean_abs_relevance"], sorted_abs["feature"],
    c=sorted_abs["mean_signed_relevance"], cmap="RdBu_r", s=60,
    edgecolors="grey", linewidths=0.4,
    norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed)
)
plt.colorbar(
    plt.cm.ScalarMappable(cmap="RdBu_r",
    norm=plt.Normalize(vmin=-vmax_signed, vmax=vmax_signed)),
    ax=ax, label="Mean Signed Relevance"
)
ax.set_xlabel("Mean Absolute Relevance", fontsize=11)
ax.set_xlim(left=0)
ax.set_title("All Features - Global LRP Relevance", fontsize=11, fontweight="bold")
ax.grid(True, axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{global_folder}/all_features_dot_plot.png", dpi=180)
plt.close()


# PER CLASS IMPORTANCE
class_importance = {}
per_class_folder = os.path.join(OUTPUT_FOLDER, "class_importance")
os.makedirs(per_class_folder, exist_ok=True)

for c in sorted(np.unique(labels)):
    mask = labels == c
    rel = all_relevance[mask]
    df_class = pd.DataFrame({
        "feature": feature_names,
        "mean_signed_relevance": rel.mean(axis=0),
        "mean_abs_relevance": np.abs(rel).mean(axis=0),
        "std_relevance": np.abs(rel).std(axis=0),
    }).sort_values("mean_abs_relevance", ascending=False)
    df_class.to_csv(f"{per_class_folder}/class_{c}_feature_importance.csv", index=False)
    class_importance[c] = df_class

    plot_top_features(df_class, TOP_N_PER_CLASS, f"{per_class_folder}/top_features_class_{c}.png", f"Stage {c} - Top {TOP_N_PER_CLASS} Features")


# STAGE PROGRESSION IMPORTANCE CURVE
stage_feature_importance = np.zeros((n_stages, len(feature_names)))
for i, stage in enumerate(unique_stages):
    stage_feature_importance[i] = np.abs(all_relevance[labels == stage]).mean(axis=0)

top_features_indices = np.argsort(global_mean_abs)[-TOP_N_FEATURES:]
top_feature_names = [feature_names[i] for i in top_features_indices]
stage_feature_importance_top = stage_feature_importance[:, top_features_indices]

fig, ax = plt.subplots(figsize=(12, 6))
cmap_prog = plt.colormaps['tab10'].resampled(len(top_feature_names))
for i, feat_name in enumerate(top_feature_names):
    ax.plot(
        unique_stages, stage_feature_importance_top[:, i],
        marker='o', label=feat_name, color=cmap_prog(i), linewidth=2
    )
ax.set_xticks(unique_stages)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Mean Absolute Relevance", fontsize=11)
ax.set_title(f"Stage Progression of Top {TOP_N_FEATURES} Features", fontsize=12, fontweight="bold")
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FOLDER, "stage_progression_importance_curve.png"), dpi=180)
plt.close()


# CORRECT vs MISCLASSIFIED
correct_folder = os.path.join(OUTPUT_FOLDER, "correct_vs_misclassified")
os.makedirs(correct_folder, exist_ok=True)
 
correct_rel = np.abs(all_relevance[correct_mask]).mean(axis=0)
incorrect_rel = np.abs(all_relevance[~correct_mask]).mean(axis=0)
diff_rel = correct_rel - incorrect_rel
top_diff_idx  = np.argsort(np.abs(diff_rel))[-TOP_N_GLOBAL:]
top_diff_names = [feature_names[i] for i in top_diff_idx]
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
axes[0].barh(top_diff_names,  correct_rel[top_diff_idx],   color="#55A868", label="Correct")
axes[0].barh(top_diff_names, -incorrect_rel[top_diff_idx], color="#C44E52", label="Misclassified")
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_xlabel("Mean Absolute Relevance")
axes[0].set_title("Feature Relevance: Correct vs Misclassified", fontweight="bold")
axes[0].invert_yaxis()
axes[1].barh(
    top_diff_names, diff_rel[top_diff_idx],
    color=["#55A868" if d > 0 else "#C44E52" for d in diff_rel[top_diff_idx]]
)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel("Δ Relevance (Correct - Misclassified)")
axes[1].set_title("Relevance Difference", fontweight="bold")
plt.suptitle(f"LRP Relevance: Correctly vs Incorrectly Classified Patients", fontsize=12, fontweight="bold")
fig.legend(handles=[
    mpatches.Patch(color="#55A868", label="Correct"),
    mpatches.Patch(color="#C44E52", label="Misclassified"),
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.04))
plt.tight_layout()
plt.savefig(os.path.join(correct_folder, "correct_vs_misclassified_relevance.png"), dpi=180, bbox_inches='tight')
plt.close()
 
pd.DataFrame({
    "feature": feature_names, "delta": diff_rel,
    "correct_rel": correct_rel, "incorrect_rel": incorrect_rel
}).sort_values("delta", ascending=False).to_csv(
    os.path.join(correct_folder, "correct_vs_misclassified_relevance.csv"), index=False
)


# LAYER SPECIALIZATION ENTROPY
layer_arrays = {name: np.array(vals) for name, vals in layer_outputs.items()}
layer_names = list(layer_arrays.keys())

entropy_folder = os.path.join(OUTPUT_FOLDER, "layer_relevance", "entropy")
os.makedirs(entropy_folder, exist_ok=True)

entropy_per_stage = {name: [] for name in layer_names}
for name, arr in layer_arrays.items():
    for stage in unique_stages:
        stage_mean = np.abs(arr[labels == stage]).mean(axis=0)
        prob = stage_mean / (stage_mean.sum() + 1e-8)
        entropy_per_stage[name].append(scipy_entropy(prob + 1e-10))

x = np.arange(n_stages)
width = 0.8 / len(layer_names)
cmap_layers = plt.colormaps['Set2'].resampled(len(layer_names))

fig, ax = plt.subplots(figsize=(9, 5))
for i, name in enumerate(layer_names):
    offset = (i - len(layer_names) / 2 + 0.5) * width
    ax.bar(
        x + offset, entropy_per_stage[name], width=width, linewidth=0.5,
        label=name.replace("_", " "), color=cmap_layers(i), edgecolor='white'
    )
ax.set_xticks(x)
ax.set_xticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xlabel("Parkinson Stage", fontsize=11)
ax.set_ylabel("Relevance Entropy", fontsize=11)
ax.set_title("Relevance Entropy per Layer per Stage", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(entropy_folder, "layer_specialization_entropy.png"), dpi=180)
plt.close()

entropy_df = pd.DataFrame(entropy_per_stage, index=[f"Stage {s}" for s in unique_stages])
entropy_df.index.name = "stage"
entropy_df.columns = [c.replace("_", " ") for c in entropy_df.columns]
entropy_df.to_csv(os.path.join(entropy_folder, "layer_specialization_entropy.csv"))



class_dfs = {
    c: pd.read_csv(os.path.join(OUTPUT_FOLDER, "class_importance", f"class_{c}_feature_importance.csv"))
    for c in unique_stages
}
 
# FEATURE VS STAGE RELEVANCE
fs_folder = os.path.join(OUTPUT_FOLDER, "feature_stage_heatmap")
os.makedirs(fs_folder, exist_ok=True)
 
top10_per_stage = set()
for df in class_dfs.values():
    top10_per_stage.update(df.nlargest(10, "mean_abs_relevance")["feature"].tolist())
top10_per_stage = sorted(top10_per_stage)
 
heatmap_data = np.zeros((n_stages, len(top10_per_stage)))
for si, stage in enumerate(unique_stages):
    for fi, feat in enumerate(top10_per_stage):
        row = class_dfs[stage][class_dfs[stage]["feature"] == feat]
        heatmap_data[si, fi] = float(row["mean_abs_relevance"].values[0]) if len(row) else 0.0
 
col_max = heatmap_data.max(axis=0)
col_max[col_max == 0] = 1
heatmap_norm = heatmap_data / col_max
 
fig, ax = plt.subplots(figsize=(max(14, len(top10_per_stage) * 0.7), 5))
im = ax.imshow(heatmap_norm, aspect='auto', cmap='YlOrBr', vmin=0, vmax=1, interpolation='nearest')
plt.colorbar(im, ax=ax, label="Normalized Relevance")
ax.set_yticks(range(n_stages))
ax.set_yticklabels([f"Stage {s}" for s in unique_stages], fontsize=11)
ax.set_xticks(range(len(top10_per_stage)))
ax.set_xticklabels(top10_per_stage, rotation=45, ha='right', fontsize=9)
for si in range(n_stages):
    for fi in range(len(top10_per_stage)):
        ax.text(fi, si, f"{heatmap_data[si, fi]:.2f}", ha='center', va='center',
                fontsize=7, color='black' if heatmap_norm[si, fi] < 0.6 else 'white')
ax.set_title("Feature vs Stage Relevance Heatmap", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(fs_folder, "feature_stage_heatmap.png"), dpi=180, bbox_inches='tight')
plt.close()
 
heatmap_raw_df = pd.DataFrame(heatmap_data, index=[f"Stage {s}" for s in unique_stages], columns=top10_per_stage)
heatmap_raw_df.index.name = "stage"
heatmap_raw_df.to_csv(os.path.join(fs_folder, "feature_stage_heatmap_raw.csv"))

heatmap_norm_df = pd.DataFrame(heatmap_norm, index=[f"Stage {s}" for s in unique_stages], columns=top10_per_stage)
heatmap_norm_df.index.name = "stage"
heatmap_norm_df.to_csv(os.path.join(fs_folder, "feature_stage_heatmap_normalized.csv"))
 
 
# FEATURE PUSH-PULL
pp_folder = os.path.join(OUTPUT_FOLDER, "feature_push_or_pull")
os.makedirs(pp_folder, exist_ok=True)
 
fig = plt.figure(figsize=(18, 11))
grid = fig.add_gridspec(2, 6, hspace=0.2, wspace=0.7)
axes = [
    fig.add_subplot(grid[0, 0:2]),
    fig.add_subplot(grid[0, 2:4]),
    fig.add_subplot(grid[0, 4:6]),
    fig.add_subplot(grid[1, 1:3]),
    fig.add_subplot(grid[1, 3:5]),
]
pp_rows = []
for ax, stage in zip(axes, unique_stages):
    top = class_dfs[stage].nlargest(TOP_N_STAGE, "mean_abs_relevance")
    colors = ["#55A868" if v > 0 else "#C44E52" for v in top["mean_signed_relevance"]]
    ax.barh(top["feature"], top["mean_signed_relevance"], color=colors, edgecolor="white", linewidth=0.3)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.invert_yaxis()
    ax.set_title(f"Stage {stage}", fontsize=11, fontweight="bold", color="black")
    ax.set_xlabel("Mean Signed Relevance", fontsize=9)
    ax.tick_params(axis='y', labelsize=8, pad=2)
    ax.yaxis.set_tick_params(length=0)
    ax.grid(True, axis='x', linestyle='--', alpha=0.4)
    for _, row in top.iterrows():
        pp_rows.append({
            "stage": stage, "feature": row["feature"],
            "mean_signed_relevance": row["mean_signed_relevance"],
            "mean_abs_relevance":    row["mean_abs_relevance"],
            "direction": "positive" if row["mean_signed_relevance"] >= 0 else "negative"
        })
 
fig.legend(handles=[
    mpatches.Patch(color="#55A868", label="Positive — confirms diagnosis"),
    mpatches.Patch(color="#C44E52", label="Negative — works against diagnosis"),
], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, 0.01))
plt.suptitle(f"Top {TOP_N_STAGE} Features per Stage - Push or Pull", fontsize=14, fontweight="bold")
plt.savefig(os.path.join(pp_folder, "feature_push_or_pull_per_stage.png"), dpi=180, bbox_inches='tight')
plt.close()
 
pd.DataFrame(pp_rows).to_csv(os.path.join(pp_folder, "feature_push_or_pull_per_stage.csv"), index=False)


print("\nLRP analysis complete. Results saved in:", OUTPUT_FOLDER)

d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\lrp.py:207: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(input_tuple)
d:\USTHB\M2\Stage Parkinson\Parkinson-prediction\.venv\Lib\site-packages\captum\attr\_core\layer\layer_lrp.py:233: UserWarning: Input Tensor 0 did not already require gradients, required_grads has been set automatically.
  gradient_mask = apply_gradient_requirements(inputs_tuple)



LRP analysis complete. Results saved in: lrp_analysis_30_v8
